In [1]:
import os

base = "/home/jjh0709/gitrepo/VISION-Instance-Seg/data_augmented"

print("=" * 45)
print(f"{'카테고리':<15} {'gen_ai 이미지 수':>15}")
print("=" * 45)

total = 0
for cat in sorted(os.listdir(base)):
    gen_dir = os.path.join(base, cat, "gen_ai", "images")
    if os.path.isdir(gen_dir):
        count = len([f for f in os.listdir(gen_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    else:
        count = 0
    total += count
    print(f"{cat:<15} {count:>15}장")

print("=" * 45)
print(f"{'합계':<15} {total:>15}장")

카테고리               gen_ai 이미지 수
Cable                       104장
Capacitor                     0장
Casting                     193장
Console                     187장
Cylinder                     27장
Electronics                   0장
Groove                        0장
Hemisphere                    0장
Lens                          0장
PCB_1                         0장
PCB_2                         0장
Ring                          0장
Screw                       256장
Wood                          0장
합계                          767장


# Cylinder Porosity vs RCS 시각화\n\n- **왼쪽**: AI 생성 이미지 (Cylinder_Porosity)\n- **오른쪽**: 원본 데이터 RCS (bbox + segmentation from COCO annotation)

In [ ]:
import json
import glob
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.collections import PatchCollection
from PIL import Image

# Paths
AI_POROSITY_DIR = "/home/jjh0709/gitrepo/VISION-Instance-Seg/scripts/augmentation/vision_ai_generated/Cylinder/Cylinder_Porosity"
AI_RCS_DIR = "/home/jjh0709/gitrepo/VISION-Instance-Seg/scripts/augmentation/vision_ai_generated/Cylinder/Cylinder_RCS"
TRAIN_DIR = "/home/jjh0709/gitrepo/VISION-Instance-Seg/data/Cylinder/train"
ANNO_PATH = f"{TRAIN_DIR}/_annotations.coco.json"

# Load COCO annotations
with open(ANNO_PATH) as f:
    coco = json.load(f)

cat_map = {c['id']: c['name'] for c in coco['categories']}
img_map = {im['id']: im for im in coco['images']}

# Group annotations by category
porosity_annos = {}  # image_id -> [annos]
rcs_annos = {}

for ann in coco['annotations']:
    cat_name = cat_map[ann['category_id']]
    if cat_name == 'Porosity':
        porosity_annos.setdefault(ann['image_id'], []).append(ann)
    elif cat_name == 'RCS':
        rcs_annos.setdefault(ann['image_id'], []).append(ann)

# AI generated image lists
ai_porosity_imgs = sorted(glob.glob(f"{AI_POROSITY_DIR}/*.png"))
ai_rcs_imgs = sorted(glob.glob(f"{AI_RCS_DIR}/*.png"))

print(f"AI Porosity 이미지: {len(ai_porosity_imgs)}장")
print(f"AI RCS 이미지: {len(ai_rcs_imgs)}장")
print(f"원본 Porosity annotations: {len(porosity_annos)} images, {sum(len(v) for v in porosity_annos.values())} annos")
print(f"원본 RCS annotations: {len(rcs_annos)} images, {sum(len(v) for v in rcs_annos.values())} annos")

In [ ]:
def draw_annotations(ax, img_path, annos, title, color):
    """Draw image with bbox and segmentation overlay."""
    img = Image.open(img_path)
    ax.imshow(img)
    
    for ann in annos:
        # Draw bbox
        x, y, w, h = ann['bbox']
        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        
        # Draw segmentation mask
        for seg in ann['segmentation']:
            poly = np.array(seg).reshape(-1, 2)
            polygon = patches.Polygon(poly, closed=True, alpha=0.3, facecolor=color, edgecolor=color, linewidth=1.5)
            ax.add_patch(polygon)
        
        # Label
        ax.text(x, y - 5, cat_map[ann['category_id']], color='white', fontsize=9,
                fontweight='bold', bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8))
    
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')

## 1. 원본 데이터: Porosity vs RCS (bbox + segmentation)

In [ ]:
# 원본 Porosity vs RCS 비교 (각 5장)
porosity_img_ids = list(porosity_annos.keys())
rcs_img_ids = list(rcs_annos.keys())

n_rows = 5
fig, axes = plt.subplots(n_rows, 2, figsize=(20, n_rows * 6))

random.seed(42)
p_samples = random.sample(porosity_img_ids, min(n_rows, len(porosity_img_ids)))
r_samples = random.sample(rcs_img_ids, min(n_rows, len(rcs_img_ids)))

for i in range(n_rows):
    # Left: Porosity
    p_id = p_samples[i]
    p_info = img_map[p_id]
    p_path = f"{TRAIN_DIR}/{p_info['file_name']}"
    draw_annotations(axes[i, 0], p_path, porosity_annos[p_id], f"Porosity - {p_info['file_name']}", '#FF4444')
    
    # Right: RCS
    r_id = r_samples[i]
    r_info = img_map[r_id]
    r_path = f"{TRAIN_DIR}/{r_info['file_name']}"
    draw_annotations(axes[i, 1], r_path, rcs_annos[r_id], f"RCS - {r_info['file_name']}", '#4488FF')

plt.suptitle("원본 데이터: Porosity (왼쪽) vs RCS (오른쪽)", fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 2. AI 생성 이미지: Cylinder_Porosity vs Cylinder_RCS

In [ ]:
# AI 생성 Porosity vs RCS 비교 (각 5장)
n_rows = 5
fig, axes = plt.subplots(n_rows, 2, figsize=(20, n_rows * 6))

random.seed(42)
ai_p_samples = random.sample(ai_porosity_imgs, min(n_rows, len(ai_porosity_imgs)))
ai_r_samples = random.sample(ai_rcs_imgs, min(n_rows, len(ai_rcs_imgs)))

for i in range(n_rows):
    # Left: AI Porosity
    img = Image.open(ai_p_samples[i])
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"AI Porosity - {ai_p_samples[i].split('/')[-1]}", fontsize=13, fontweight='bold')
    axes[i, 0].axis('off')
    
    # Right: AI RCS
    img = Image.open(ai_r_samples[i])
    axes[i, 1].imshow(img)
    axes[i, 1].set_title(f"AI RCS - {ai_r_samples[i].split('/')[-1]}", fontsize=13, fontweight='bold')
    axes[i, 1].axis('off')

plt.suptitle("AI 생성 이미지: Porosity (왼쪽) vs RCS (오른쪽)", fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 3. 원본 vs AI 생성 비교 (같은 레이블끼리)

In [ ]:
# 원본(annotation 포함) vs AI 생성 나란히 비교
fig, axes = plt.subplots(4, 2, figsize=(20, 24))

random.seed(123)

# Row 0-1: Porosity (원본 vs AI)
p_samples2 = random.sample(porosity_img_ids, 2)
for i in range(2):
    p_id = p_samples2[i]
    p_info = img_map[p_id]
    p_path = f"{TRAIN_DIR}/{p_info['file_name']}"
    draw_annotations(axes[i, 0], p_path, porosity_annos[p_id], f"[원본] Porosity - {p_info['file_name']}", '#FF4444')
    
    ai_img = Image.open(ai_porosity_imgs[i])
    axes[i, 1].imshow(ai_img)
    axes[i, 1].set_title(f"[AI 생성] {ai_porosity_imgs[i].split('/')[-1]}", fontsize=13, fontweight='bold')
    axes[i, 1].axis('off')

# Row 2-3: RCS (원본 vs AI)
r_samples2 = random.sample(rcs_img_ids, 2)
for i in range(2):
    r_id = r_samples2[i]
    r_info = img_map[r_id]
    r_path = f"{TRAIN_DIR}/{r_info['file_name']}"
    draw_annotations(axes[i+2, 0], r_path, rcs_annos[r_id], f"[원본] RCS - {r_info['file_name']}", '#4488FF')
    
    ai_img = Image.open(ai_rcs_imgs[i])
    axes[i+2, 1].imshow(ai_img)
    axes[i+2, 1].set_title(f"[AI 생성] {ai_rcs_imgs[i].split('/')[-1]}", fontsize=13, fontweight='bold')
    axes[i+2, 1].axis('off')

plt.suptitle("원본 (왼쪽, bbox+seg) vs AI 생성 (오른쪽)", fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()